### 전처리 데이터 병합 (핫스팟용)
(1) usage_all + repair_all 매칭 <br>
(2) 대여소별 고장 집계 <br>
(3) location_all 조인 <br>
(4) EDA

### 1. 환경 설정


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_path = '/content/drive/MyDrive/26-1_BITAmin_TS_project'
PROCESSED_DIR = os.path.join(base_path, 'processed_data')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('base_path:', base_path)
print('processed_dir:', PROCESSED_DIR)


### 2. usage / repair 로드


In [ ]:
usage_path_candidates = [
    os.path.join(PROCESSED_DIR, 'usage_all.parquet'),
]
repair_path_candidates = [
    os.path.join(PROCESSED_DIR, 'repair_all.parquet'),
]

def pick_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

usage_path = pick_existing(usage_path_candidates)
repair_path = pick_existing(repair_path_candidates)

assert usage_path is not None, f'usage 파일 없음: {usage_path_candidates}'
assert repair_path is not None, f'repair 파일 없음: {repair_path_candidates}'

usage_raw = pd.read_parquet(usage_path)
repair_raw = pd.read_parquet(repair_path)

print('usage path :', usage_path)
print('repair path:', repair_path)
print('usage shape:', usage_raw.shape)
print('repair shape:', repair_raw.shape)
display(usage_raw.head())
display(repair_raw.head())


### 3. usage 정리 (merge 키 기준)


In [ ]:
usage_need = ['자전거번호', '반납일시', '반납대여소번호']
missing_usage = [c for c in usage_need if c not in usage_raw.columns]
assert len(missing_usage) == 0, f'usage 필수 컬럼 누락: {missing_usage}'

usage_m = usage_raw[usage_need].copy()
usage_m['자전거번호'] = usage_m['자전거번호'].astype('string').str.strip().str.upper()
usage_m['반납일시'] = pd.to_datetime(usage_m['반납일시'], errors='coerce')
usage_m['반납대여소번호'] = (
    usage_m['반납대여소번호'].astype('string').str.strip()
    .str.replace(r'\.0$', '', regex=True)
    .str.replace(r'[^0-9]', '', regex=True)
    .replace({'': pd.NA})
)
usage_m = usage_m.dropna(subset=['자전거번호', '반납일시', '반납대여소번호'])
usage_m = usage_m.sort_values(['자전거번호', '반납일시']).reset_index(drop=True)

print('usage_m shape:', usage_m.shape)


### 4. repair 정리 (merge 키 기준)


In [ ]:
repair_df = repair_raw.copy()
repair_df.columns = repair_df.columns.str.strip()

if '구분' in repair_df.columns and '고장구분' not in repair_df.columns:
    repair_df = repair_df.rename(columns={'구분': '고장구분'})

for c in ['자전거번호', '등록일시', '고장구분']:
    if c not in repair_df.columns:
        repair_df[c] = pd.NA

repair_m = repair_df[['자전거번호', '등록일시', '고장구분']].copy()
repair_m['자전거번호'] = repair_m['자전거번호'].astype('string').str.strip().str.upper()
reg = repair_m['등록일시'].astype('string').str.strip()
reg = reg.str.replace('.', '-', regex=False).str.replace('/', '-', regex=False)
reg = reg.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
repair_m['등록일시'] = pd.to_datetime(reg, errors='coerce')
repair_m['고장구분'] = repair_m['고장구분'].astype('string').str.strip()

repair_m = repair_m.dropna(subset=['자전거번호', '등록일시'])
repair_m = repair_m.sort_values(['자전거번호', '등록일시']).reset_index(drop=True)

print('repair_m shape:', repair_m.shape)


### 5. usage + repair 매칭


In [ ]:
usage_bikes = set(usage_m['자전거번호'].dropna().unique())
repair_bikes = set(repair_m['자전거번호'].dropna().unique())
print('usage 자전거 수:', len(usage_bikes))
print('repair 자전거 수:', len(repair_bikes))
print('공통 자전거 수:', len(usage_bikes & repair_bikes))

repair_station_events = pd.merge_asof(
    repair_m,
    usage_m,
    by='자전거번호',
    left_on='등록일시',
    right_on='반납일시',
    direction='backward',
    allow_exact_matches=True,
)

repair_station_events = repair_station_events.rename(columns={'반납대여소번호': '추정고장대여소ID'})
repair_station_events['시간차_분'] = (
    repair_station_events['등록일시'] - repair_station_events['반납일시']
).dt.total_seconds() / 60
repair_station_events['위치매핑성공'] = repair_station_events['추정고장대여소ID'].notna()

print('repair_station_events shape:', repair_station_events.shape)
print('위치 매핑 성공률(%):', round(repair_station_events['위치매핑성공'].mean() * 100, 4))
print('음수 시간차 건수:', int((repair_station_events['시간차_분'] < 0).fillna(False).sum()))
display(repair_station_events.head())


### 5.1 EDA: 매칭 품질


In [ ]:
cnt = repair_station_events['위치매핑성공'].value_counts(dropna=False)
plt.figure(figsize=(5,3))
plt.bar(cnt.index.astype(str), cnt.values)
plt.title('위치 매핑 성공/실패 건수')
plt.xlabel('위치매핑성공')
plt.ylabel('건수')
plt.tight_layout()
plt.show()

tmp = repair_station_events['시간차_분'].dropna()
tmp = tmp[(tmp >= 0) & (tmp <= tmp.quantile(0.99))]
plt.figure(figsize=(6,3))
plt.hist(tmp, bins=50)
plt.title('시간차(분) 분포 (상위 1% 제외)')
plt.xlabel('분')
plt.ylabel('빈도')
plt.tight_layout()
plt.show()


### 6. 매칭 결과 저장


In [ ]:
repair_station_events_path = os.path.join(PROCESSED_DIR, 'repair_station_events.parquet')
repair_station_events.to_parquet(repair_station_events_path, index=False)
print('저장 완료:', repair_station_events_path)


### 7. 대여소별 고장 핫스팟 집계


In [ ]:
fault_base = repair_station_events[repair_station_events['위치매핑성공']].copy()
fault_base['date'] = pd.to_datetime(fault_base['등록일시']).dt.floor('D')

station_hotspot = (
    fault_base
    .groupby('추정고장대여소ID', as_index=False)
    .agg(
        고장건수=('등록일시', 'count'),
        고장자전거수=('자전거번호', 'nunique'),
        고장유형수=('고장구분', 'nunique'),
        첫고장일=('등록일시', 'min'),
        마지막고장일=('등록일시', 'max')
    )
    .sort_values('고장건수', ascending=False)
    .reset_index(drop=True)
)

station_hotspot_daily = (
    fault_base
    .groupby(['date', '추정고장대여소ID'], as_index=False)
    .agg(고장건수=('등록일시', 'count'))
    .sort_values(['date', '고장건수'], ascending=[True, False])
    .reset_index(drop=True)
)

print('station_hotspot shape:', station_hotspot.shape)
print('station_hotspot_daily shape:', station_hotspot_daily.shape)
display(station_hotspot.head(10))


### 7.1 EDA: 핫스팟/월별 추이


In [ ]:
# Top 20 대여소
plot_df = station_hotspot.head(20)
plt.figure(figsize=(10,4))
plt.bar(plot_df['추정고장대여소ID'].astype(str), plot_df['고장건수'])
plt.title('고장 핫스팟 TOP 20')
plt.xlabel('대여소ID')
plt.ylabel('고장건수')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# 월별 고장 추이
monthly = station_hotspot_daily.copy()
monthly['year_month'] = pd.to_datetime(monthly['date']).dt.to_period('M').astype(str)
monthly = monthly.groupby('year_month', as_index=False)['고장건수'].sum()

plt.figure(figsize=(10,4))
plt.plot(monthly['year_month'], monthly['고장건수'], marker='o')
plt.title('월별 고장건수 추이')
plt.xlabel('year_month')
plt.ylabel('고장건수')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 8. 핫스팟 집계 저장


In [ ]:
station_hotspot_path = os.path.join(PROCESSED_DIR, 'station_hotspot.parquet')
station_hotspot_daily_path = os.path.join(PROCESSED_DIR, 'station_hotspot_daily.parquet')

station_hotspot.to_parquet(station_hotspot_path, index=False)
station_hotspot_daily.to_parquet(station_hotspot_daily_path, index=False)

print('저장 완료:')
print('-', station_hotspot_path)
print('-', station_hotspot_daily_path)


### 9. 위치 데이터 조인


In [ ]:
loc_path_candidates = [
    os.path.join(PROCESSED_DIR, 'location_all.parquet'),
]

loc_path = pick_existing(loc_path_candidates)
if loc_path is None:
    print('location 파일 없음:', loc_path_candidates)
else:
    loc = pd.read_parquet(loc_path).copy()
    loc.columns = [c.strip() for c in loc.columns]

    # location_all 기준: 대여소번호 / 위도 / 경도
    if '대여소번호' in loc.columns:
        loc = loc[['대여소번호', '위도', '경도']].copy()
        loc = loc.rename(columns={'대여소번호': '추정고장대여소ID'})
    else:
        loc = loc[['추정고장대여소ID', '위도', '경도']].copy()

    loc['추정고장대여소ID'] = (
        loc['추정고장대여소ID'].astype('string').str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.replace(r'[^0-9]', '', regex=True)
        .replace({'': pd.NA})
    )

    station_hotspot_map = station_hotspot.merge(loc, on='추정고장대여소ID', how='left')
    station_hotspot_map['좌표매핑성공'] = station_hotspot_map['위도'].notna() & station_hotspot_map['경도'].notna()

    out_path = os.path.join(PROCESSED_DIR, 'station_hotspot_with_coords.parquet')
    station_hotspot_map.to_parquet(out_path, index=False)

    print('location path:', loc_path)
    print('지도용 저장:', out_path)
    print('좌표 매핑률(%):', round(station_hotspot_map['좌표매핑성공'].mean() * 100, 4))
    display(station_hotspot_map.head())
